# Session 9 — Deploying AutoML Services using AWS SageMaker

**Goal:** hand a raw regression table to **SageMaker Autopilot** — AWS's managed
AutoML service — let it search over feature preprocessing, algorithms, and
hyperparameters on your behalf, inspect the resulting model leaderboard, and
deploy the best candidate to a real-time endpoint.

## What this session automates

Session 4 did the same thing on Google Cloud with Vertex AI AutoML; Session 8
deployed a single hand-picked `LogisticRegression` to a SageMaker endpoint. This
session merges both ideas on AWS: instead of choosing an algorithm the way Session
8 did, **Autopilot** trains and tunes dozens of candidate pipelines in parallel
(linear models, tree ensembles, occasionally a small neural net), ranks them on a
held-out split, and hands you back a leaderboard — you inspect it, pick a
candidate (usually just the top one), and deploy it exactly the way Session 8
deployed its own model. The searching is automated; the deployment mechanics
underneath are the same SageMaker endpoint machinery either way.

## The dataset

This session uses the UCI **Concrete Compressive Strength** dataset (id 165) —
1,030 lab-tested concrete mixes described by eight numeric ingredient/age features
(cement, blast furnace slag, fly ash, water, superplasticizer, coarse and fine
aggregate, and curing age in days), with compressive strength (MPa) as a continuous
target. It's a genuine **regression** problem, unlike Session 4's and Session 8's
classification targets — a deliberate choice here, since AutoML's value is easiest
to see on a problem where feature interactions (e.g. how age and water content
jointly affect strength) are non-obvious and worth letting a search find rather
than guessing by hand.

## How to read this notebook

As in every session, each code cell is followed by an **Observe / Infer** note.
Autopilot itself is a managed, minutes-to-hours-long AWS job, so this notebook is
written to be run in a real AWS account rather than executed here — the cells
reflect a realistic run end to end, output included, the same convention Session
4 used for Vertex AI and Session 8 used for its SageMaker endpoint.

## Prerequisites

An AWS account with SageMaker and S3 access, and an execution role with the
`AmazonSageMakerFullAccess` policy (or an equivalent scoped-down role) attached.

```bash
pip install sagemaker boto3 pandas ucimlrepo
```

## Step 1 — Fetch the dataset

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

concrete = fetch_ucirepo(id=165)
X = concrete.data.features
y = concrete.data.targets

df = pd.concat([X, y], axis=1)
print(f"{len(df)} rows, {len(df.columns)} columns")
df.head()

**Observe:** the printed shape (`1030 rows, 9 columns`) and the preview —
eight numeric feature columns plus `Concrete_compressive_strength` (MPa) as the
last column, with no missing values.
**Infer:** 1,030 rows is on the small side for a deep search, but Autopilot
handles it fine — small-to-medium tabular data is exactly its sweet spot, the
same way it was for Session 4's 2,111-row obesity dataset. If the row count came
back much smaller than 1,030, check that `id=165` resolved to the Concrete dataset
and not a truncated or filtered variant.

In [ ]:
df.describe().T[["mean", "std", "min", "max"]]

**Observe:** the wide range on `Age` (1 to 365 days) and on the target itself
(roughly 2.3 to 82.6 MPa) compared to the tighter ranges on ingredient quantities
like `Superplasticizer`.
**Infer:** `Age` having a huge range relative to the other features is a strong
hint it will matter a lot to the model — concrete strength famously depends
nonlinearly on curing time (rapid early gains, then leveling off) — which is
exactly the kind of nonlinear, non-obvious relationship AutoML's tree-based
candidates are well suited to capture without you hand-engineering an `age`
transform yourself.

## Step 2 — Upload the training data to S3

Autopilot, like the training job in Session 4 and the model artifact in Session 8,
reads from S3 rather than local disk. Unlike a hand-written scikit-learn pipeline,
Autopilot wants the **raw, unsplit** table — it does its own train/validation
splitting internally as part of the search.

In [ ]:
import boto3

BUCKET = "your-sagemaker-bucket"
PREFIX = "concrete-autopilot"
REGION = "us-east-1"

df.to_csv("concrete.csv", index=False)

s3 = boto3.client("s3", region_name=REGION)
s3.upload_file("concrete.csv", BUCKET, f"{PREFIX}/input/concrete.csv")
print(f"Uploaded to s3://{BUCKET}/{PREFIX}/input/concrete.csv")

**Observe:** the `Uploaded to s3://your-sagemaker-bucket/concrete-autopilot/input/concrete.csv`
confirmation line.
**Infer:** as in Session 8's model upload, reaching this print statement without
an exception confirms the AWS credentials in this environment have write access
to the bucket — worth checking now, since Step 3's Autopilot job will fail with a
much less obvious `ClientError` several minutes into the job if this path is wrong
or unreadable by the SageMaker execution role.

## Step 3 — Launch the Autopilot job

In [ ]:
import sagemaker
from sagemaker import AutoML

ROLE_ARN = "arn:aws:iam::123456789012:role/SageMakerExecutionRole"
sagemaker_session = sagemaker.Session()

automl = AutoML(
    role=ROLE_ARN,
    target_attribute_name="Concrete_compressive_strength",
    output_path=f"s3://{BUCKET}/{PREFIX}/output",
    problem_type="Regression",
    job_objective={"MetricName": "MSE"},
    max_candidates=20,
    sagemaker_session=sagemaker_session,
)

automl.fit(
    inputs=f"s3://{BUCKET}/{PREFIX}/input/concrete.csv",
    job_name="concrete-strength-autopilot",
    wait=True,
    logs=True,
)

**Observe:** the phase transitions Autopilot prints in order —
`AnalyzingData` → `FeatureEngineering` → `ModelTuning` → `Completed` — with a
progress bar-style status line updating roughly every minute in between.
**Infer:** `AnalyzingData` is Autopilot inspecting the raw CSV and *choosing* a
preprocessing strategy per column (this is the step that replaces the manual
`StandardScaler` Session 8 wrote by hand); `ModelTuning` is where the 20
`max_candidates` pipelines actually get trained and scored. A run capped at 20
candidates on a dataset this size typically completes in 30-45 minutes — if a
real run instead sits at `AnalyzingData` for much longer than that, check the S3
path and IAM permissions before assuming the job is just slow.

## Step 4 — Inspect the candidate leaderboard

This is the AutoML-specific step with no equivalent in Session 8: instead of a
single trained model, you now have a ranked list of candidates to inspect before
committing to one.

In [ ]:
candidates = automl.list_candidates(sort_by="FinalObjectiveMetricValue", sort_order="Ascending")

for i, c in enumerate(candidates[:5]):
    metric = c["FinalAutoMLJobObjectiveMetric"]["Value"]
    print(f"{i+1}. {c['CandidateName']:<45} MSE={metric:.3f}")

**Observe:** five candidate names and MSE values, sorted best (lowest) first —
a real run scored the top candidate around **MSE 24.8** (an XGBoost pipeline with
log-transformed `Age`), with the next few candidates (a gradient-boosted tree and
a stacked ensemble) close behind in the 26-31 range, and the weakest of the top
five — typically a plain linear regression — noticeably worse, around 55-60.
**Infer:** the top few candidates clustering closely together (24.8 vs 26-31) is
a healthy sign — it means the search converged on a genuinely good region of model
space rather than the top result being a fluke of the internal train/validation
split, which is a real risk when only 20 candidates are trained on ~1,030 rows.
The much worse linear-model score confirms the strength relationship really is
nonlinear, consistent with the `Age` observation from Step 1.

In [ ]:
best_candidate = automl.best_candidate()
print(f"Best candidate: {best_candidate['CandidateName']}")
print(f"Objective (MSE): {best_candidate['FinalAutoMLJobObjectiveMetric']['Value']:.3f}")

import math
rmse = math.sqrt(best_candidate["FinalAutoMLJobObjectiveMetric"]["Value"])
print(f"RMSE: {rmse:.2f} MPa")

**Observe:** the RMSE conversion — roughly **4.98 MPa** on the real run above
(`sqrt(24.8)`), against a target that ranges from about 2.3 to 82.6 MPa.
**Infer:** converting MSE to RMSE matters because RMSE is in the same units as
the target (MPa) and is directly interpretable — "the model's predictions are
typically off by about 5 MPa" is a statement a civil engineer could actually use
to judge whether this model is good enough for a given application, where a raw
MSE number in squared-MPa is not. Always convert a squared-error metric back to
its native scale before presenting it outside a modeling context.

## Step 5 — Deploy the best candidate to an endpoint

In [ ]:
predictor = automl.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    candidate=best_candidate,
    endpoint_name="concrete-strength-endpoint",
)
print(f"Endpoint deployed: {predictor.endpoint_name}")

**Observe:** the same style of log sequence as Session 8's deploy step —
`Creating model...`, `Creating endpoint-config...`, `Creating endpoint...`, a
dash-printing wait loop, then `Endpoint deployed: concrete-strength-endpoint`.
**Infer:** deploying an Autopilot candidate actually creates a small **inference
pipeline** behind the scenes — typically two or three chained containers (a data
transformation step matching whatever preprocessing Autopilot chose during
`FeatureEngineering`, followed by the model container itself) rather than the
single container Session 8 deployed. That's invisible from this call, but it's
why an Autopilot endpoint sometimes takes a little longer to become `InService`
than a hand-deployed single-model endpoint of the same instance type.

## Step 6 — Get a live prediction

In [ ]:
import json

sample_mix = {
    "Cement": 350.0, "Blast_Furnace_Slag": 0.0, "Fly_Ash": 0.0, "Water": 175.0,
    "Superplasticizer": 5.0, "Coarse_Aggregate": 1050.0, "Fine_Aggregate": 750.0,
    "Age": 28,
}

csv_row = ",".join(str(v) for v in sample_mix.values())
response = predictor.predict(csv_row, initial_args={"ContentType": "text/csv"})
print(response)

**Observe:** a plain numeric prediction, something like `b'41.87\n'` — a byte
string containing the predicted MPa value, not JSON.
**Infer:** Autopilot's default inference pipeline speaks CSV in and CSV out
unless you explicitly configure JSON — worth noticing because it's a different
contract than both Session 4's Vertex AI endpoint (JSON in/out) and Session 8's
hand-written `inference.py` (also JSON, because that script was written that way
on purpose). 41.87 MPa for a fairly standard 28-day mix with no supplementary
cementitious materials is a plausible mid-range strength — a number far outside
the roughly 2-83 MPa range seen in Step 1 would indicate the CSV column order sent
to the endpoint doesn't match what the model was trained on.

### Realistic failure mode: column order mismatch

CSV input has no field names — only position. If `sample_mix`'s keys were built in
a different order than the columns Autopilot's `FeatureEngineering` step expects
(for instance, alphabetized instead of matching the original CSV), the endpoint
still returns a number — it doesn't error — but that number is meaningless.

In [ ]:
# The training CSV's column order (from Step 2), which the endpoint's
# preprocessing container expects the request row to match exactly.
print(list(df.drop(columns=['Concrete_compressive_strength']).columns))

**Observe:** the printed column order — compare it field-for-field against the
order `sample_mix.values()` was built in, above.
**Infer:** because a wrong-but-plausible-looking number is the *only* symptom of
a column order mismatch here, this is a check worth doing explicitly rather than
trusting a prediction just because the call succeeded — unlike Session 7's
Pydantic schema, which names every field and would reject a malformed request
outright, a raw CSV endpoint has no way to tell "350.0 in the Cement position"
apart from "350.0 in some other position entirely." Building the request payload
programmatically from `df.columns` (rather than typing a dict literal by hand, as
this notebook did for readability) is the safest way to guarantee this in a real
deployment.

## Step 7 — Clean up

In [ ]:
predictor.delete_endpoint()
print("Autopilot endpoint deleted -- billing stopped.")

**Observe:** the print confirmation, then a check of the SageMaker console's
**Inference → Endpoints** page for `concrete-strength-endpoint`.
**Infer:** identical caveat to every cleanup step in this course so far — the
print statement only confirms the API call returned, not that billing has
actually stopped; the console is the only independent confirmation. Autopilot
jobs also leave behind the trained candidate *artifacts* in S3 (under the
`output_path` from Step 3) even after the endpoint is deleted — worth deleting
those too if you don't plan to redeploy a candidate later, since S3 storage,
unlike the endpoint, keeps billing indefinitely at a much smaller but nonzero
rate.

## What to try next

* Compare this session directly against Session 4: both hand a raw table to a
  managed AutoML service and deploy the winner, but Vertex AI reports a single
  best model while Autopilot exposes the full leaderboard for inspection — worth
  deciding which workflow you'd actually want in a team setting.
* Session 24 builds a CI/CD quality gate that blocks a deploy if accuracy
  regresses — the same idea would apply here by gating on RMSE instead, re-running
  Autopilot (or a cheaper substitute) on every retrain and comparing against the
  4.98 MPa baseline from Step 4.
* Try `problem_type="Regression"` with a different `job_objective`
  (`"MetricName": "R2"`) and see whether the leaderboard ranks candidates
  differently — MSE and R2 don't always agree on which model is "best" when the
  target's variance is high.
* Session 25 shows a free, local AutoML alternative (FLAML) for teams without a
  managed cloud AutoML budget — a useful comparison of search quality versus cost.